# Experimento 5 — DFT invertida: convolução no início, multiplicação espectral no final

**Motivação.** No experimento 4 substituímos os *primeiros* blocos `DepthwiseConv2D` da MobileNetV3Large por uma FFT sem parâmetros (`|FFT(x)|`). Os resultados foram ruins, provavelmente porque:

1. As primeiras camadas codificam features de baixo nível (bordas, texturas) com forte estrutura espacial local — uma FFT global destrói essa localidade.
2. A operação `|FFT(x)|` não tem parâmetros aprendíveis, então a rede perde capacidade justamente onde mais precisa.
3. Os pesos do ImageNet foram descartados nas camadas mais sensíveis.

**Hipótese desta vez.** Mantemos as convoluções no início (estágio onde features locais e pesos ImageNet são valiosos) e substituímos os *últimos* blocos `DepthwiseConv2D` por uma **multiplicação aprendível no domínio da frequência**. Pelo teorema da convolução, multiplicar no domínio de Fourier é equivalente a uma convolução circular no domínio espacial — então o filtro espectral aprendível tem expressividade comparável à de uma convolução depthwise, mas com receptive field global de graça.

**Por que pode funcionar melhor:**
- Nas camadas profundas as feature maps são pequenas (7×7, 14×14) → FFT é barata.
- Há muitos canais (até 960) → filtros espectrais por canal têm bastante expressividade.
- O filtro complexo é inicializado a partir do kernel depthwise pré-treinado (via padded-FFT), preservando o conhecimento do ImageNet como ponto de partida e permitindo ajuste fino.

In [ ]:
import os
import time
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
import keras.backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, DepthwiseConv2D, Input
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import f1_score as sklearn_f1

print('TF version:', tf.__version__)

In [ ]:
# === Configuração ===
DISPOSITIVOS = ['fridge']
IMAGENS      = ['tbg']
BATCH        = 32
FOLDER_I     = 'pickle_data'   # mesmos dados do notebook 1
N_RUNS       = 3               # runs por combinação

# Variantes: nome -> quantos blocos DepthwiseConv2D substituir por mult. espectral.
# A substituição agora começa pelo FINAL da rede (camadas profundas).
VARIANTS = {
    'baseline':              0,
    'spectral_last_1block':  1,
    'spectral_last_3blocks': 3,
}

## Camada `SpectralMultiplyDepthwise`

Substitui um `DepthwiseConv2D` por uma multiplicação aprendível no domínio da frequência:

$$y = \mathrm{IFFT}_2 \big( W \odot \mathrm{FFT}_2(x) \big)$$

onde $W \in \mathbb{C}^{H \times W \times C}$ é o filtro complexo aprendível (um por canal — depthwise). Para preservar o conhecimento do ImageNet, $W$ é inicializado com a FFT do kernel `DepthwiseConv2D` original, com zero-padding até as dimensões espaciais da feature map.

Como pesos complexos não são suportados diretamente no Keras, usamos dois pesos reais (`W_real`, `W_imag`) que são combinados na chamada. O stride original é preservado via `AveragePooling2D` quando necessário.

In [ ]:
class SpectralMultiplyDepthwise(tf.keras.layers.Layer):
    """
    Substitui DepthwiseConv2D por multiplicação aprendível no dominio da frequencia.
    
        y = Re{ IFFT2( W * FFT2(x) ) }
    
    com W complexo aprendivel de shape (H, W, C) — depthwise (um filtro por canal).
    
    Entrada : (B, H, W, C)
    Saida   : (B, H/s, W/s, C)  onde s = stride original (via AveragePooling2D).
    
    Inicializacao: se `pretrained_kernel` for fornecido (shape do kernel DW original,
    (kh, kw, C, 1)), o filtro espectral e inicializado com a FFT do kernel padded,
    o que preserva o conhecimento do ImageNet como ponto de partida.
    """
    def __init__(self, strides=(1, 1), pretrained_kernel=None, **kwargs):
        super().__init__(**kwargs)
        self.strides = (int(strides[0]), int(strides[1]))
        # numpy array com o kernel original (kh, kw, C, 1) ou None
        self._pretrained_kernel = pretrained_kernel

    def build(self, input_shape):
        # input_shape: (B, H, W, C)
        H = int(input_shape[1])
        W = int(input_shape[2])
        C = int(input_shape[3])
        self._H, self._W, self._C = H, W, C

        # Inicializacao do filtro complexo (parte real e imag).
        if self._pretrained_kernel is not None:
            # kernel: (kh, kw, C, 1) -> (kh, kw, C)
            k = np.asarray(self._pretrained_kernel, dtype=np.float32)
            if k.ndim == 4:
                k = k[..., 0]
            kh, kw, kc = k.shape
            assert kc == C, f"Canais incompativeis: kernel C={kc}, input C={C}"
            # zero-pad ate (H, W, C) e desloca para que o centro do kernel
            # fique em (0, 0) (convolucao circular)
            padded = np.zeros((H, W, C), dtype=np.float32)
            padded[:kh, :kw, :] = k
            # roll para alinhar o centro do kernel com a origem da FFT
            padded = np.roll(padded, shift=(-(kh // 2), -(kw // 2)), axis=(0, 1))
            # FFT 2D por canal -> (H, W, C) complexo
            # tf.signal.fft2d opera nas 2 ultimas dims, entao reorganizamos
            padded_chw = np.transpose(padded, (2, 0, 1))           # (C, H, W)
            spec       = np.fft.fft2(padded_chw).astype(np.complex64)
            spec       = np.transpose(spec, (1, 2, 0))             # (H, W, C)
            init_real  = tf.constant_initializer(spec.real.astype(np.float32))
            init_imag  = tf.constant_initializer(spec.imag.astype(np.float32))
        else:
            # filtro identidade no dominio da frequencia: real=1, imag=0
            init_real = tf.constant_initializer(1.0)
            init_imag = tf.constant_initializer(0.0)

        self.W_real = self.add_weight(
            name='W_real', shape=(H, W, C),
            initializer=init_real, trainable=True,
        )
        self.W_imag = self.add_weight(
            name='W_imag', shape=(H, W, C),
            initializer=init_imag, trainable=True,
        )

        s = self.strides
        if s[0] > 1 or s[1] > 1:
            self._pool = tf.keras.layers.AveragePooling2D(
                pool_size=s, strides=s, padding='same'
            )
        else:
            self._pool = None

        # libera a referencia ao kernel pre-treinado (ja foi usado pra init)
        self._pretrained_kernel = None
        super().build(input_shape)

    def call(self, x, training=None):
        # x: (B, H, W, C) -> (B, C, H, W) para FFT2D operar em (H, W)
        x_c   = tf.cast(x, tf.complex64)
        x_chw = tf.transpose(x_c, [0, 3, 1, 2])
        X     = tf.signal.fft2d(x_chw)                            # (B, C, H, W) complexo
        X     = tf.transpose(X, [0, 2, 3, 1])                     # (B, H, W, C)
        # filtro complexo W = W_real + i * W_imag
        W = tf.complex(self.W_real, self.W_imag)                  # (H, W, C)
        Y = X * W[tf.newaxis, ...]                                # broadcast no batch
        # IFFT volta para o espaco
        Y_chw = tf.transpose(Y, [0, 3, 1, 2])                     # (B, C, H, W)
        y     = tf.signal.ifft2d(Y_chw)
        y     = tf.math.real(y)
        out   = tf.transpose(y, [0, 2, 3, 1])                     # (B, H, W, C)
        return self._pool(out) if self._pool is not None else out

    def get_config(self):
        # nao serializa o kernel pre-treinado (apos build ele e None mesmo)
        return {**super().get_config(), 'strides': self.strides}

## Construtores de modelo

Substitui os **últimos** N blocos `DepthwiseConv2D` por `SpectralMultiplyDepthwise`. Diferente do experimento 4, a contagem agora é feita primeiro (para descobrir o total) e só os blocos com `idx > total - N` são substituídos.

Os pesos do `DepthwiseConv2D` original são passados como inicialização para o filtro espectral (via padded-FFT).

In [ ]:
def _count_depthwise(base):
    return sum(1 for l in base.layers if isinstance(l, DepthwiseConv2D))


def build_feature_extractor(input_shape, n_spectral=0, name=None):
    """
    Constroi extrator de features baseado em MobileNetV3Large.

    n_spectral=0 -> MobileNetV3Large original congelado
    n_spectral=N -> ULTIMOS N blocos DepthwiseConv2D -> SpectralMultiplyDepthwise

    O bloco inteiro que contem o depthwise substituido e descongelado (expansao,
    BN, SE block, projecao), nao apenas a camada espectral. Isso permite que BNs
    e o SE block se adaptem a nova distribuicao de saida do filtro espectral.
    """
    base = MobileNetV3Large(
        input_shape=input_shape, weights='imagenet', include_top=False
    )

    if n_spectral == 0:
        for layer in base.layers:
            layer.trainable = False
        out = GlobalAveragePooling2D()(base.output)
        return Model(inputs=base.input, outputs=out,
                     name=name or 'FE_baseline')

    total_dw  = _count_depthwise(base)
    threshold = total_dw - n_spectral
    counter   = [0]
    replaced_prefixes = []  # prefixos dos blocos cujo depthwise foi substituido

    def clone_fn(layer):
        if isinstance(layer, DepthwiseConv2D):
            counter[0] += 1
            if counter[0] > threshold:
                cfg     = layer.get_config()
                strides = cfg.get('strides', (1, 1))
                pretrained_kernel = None
                if layer.built and len(layer.weights) > 0:
                    pretrained_kernel = layer.weights[0].numpy()
                # ex: 'expanded_conv_14/depthwise' -> 'expanded_conv_14'
                replaced_prefixes.append(layer.name.rsplit('/', 1)[0])
                return SpectralMultiplyDepthwise(
                    strides=strides,
                    pretrained_kernel=pretrained_kernel,
                    name=f'smdw_{counter[0]}'
                )
        return layer

    cloned = tf.keras.models.clone_model(base, clone_function=clone_fn)

    # clone_model nao copia pesos; copiar manualmente para camadas nao-espectrais
    for base_layer in base.layers:
        try:
            cloned_layer = cloned.get_layer(base_layer.name)
            weights = base_layer.get_weights()
            if weights:
                cloned_layer.set_weights(weights)
        except ValueError:
            pass  # camada substituida por SpectralMultiplyDepthwise

    # congela toda a backbone; desbloqueia apenas os blocos modificados inteiros
    # (expansao + BN + SE + projecao + camada espectral)
    for layer in cloned.layers:
        layer.trainable = (
            isinstance(layer, SpectralMultiplyDepthwise)
            or any(layer.name.startswith(prefix) for prefix in replaced_prefixes)
        )

    out = GlobalAveragePooling2D()(cloned.output)
    return Model(inputs=cloned.input, outputs=out,
                 name=name or f'FE_spectral_last_{n_spectral}')


def build_full_model(input_shape, n_spectral=0):
    """Feature extractor + cabeca MLP identica ao notebook 1."""
    fe = build_feature_extractor(input_shape, n_spectral)
    x  = Dense(64, activation='relu')(fe.output)
    x  = Dropout(0.25)(x)
    x  = Dense(64, activation='relu')(x)
    x  = Dropout(0.25)(x)
    out = Dense(1, activation='sigmoid')(x)
    model = Model(inputs=fe.input, outputs=out,
                  name=f'model_spectral_last_{n_spectral}')
    model.compile(
        loss='binary_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )
    return model

## Utilitários de benchmark

In [ ]:
def model_size_mb(model):
    """Tamanho estimado em MB (parametros float32)."""
    n_params = sum(tf.size(w).numpy() for w in model.weights)
    return n_params * 4 / (1024 ** 2)


def count_params(model):
    trainable     = sum(tf.size(w).numpy() for w in model.trainable_weights)
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    return trainable, non_trainable


def measure_latency_ms(model, x_sample, n_warmup=5, n_reps=20):
    """Latencia mediana de inferencia em ms (batch completo)."""
    for _ in range(n_warmup):
        _ = model(x_sample, training=False)
    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        _ = model(x_sample, training=False)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))


def load_data(disp, img, folder=FOLDER_I):
    """Carrega splits train/val/test - mesma divisao 60/20/20 do notebook 1."""
    load = lambda fname: pickle.load(open(fname, 'rb'))
    X_tr = load(f"{folder}/X_{img}_train({disp}).pickle")
    y_tr = load(f"{folder}/y_train({disp}).pickle")
    X_va = load(f"{folder}/X_{img}_val({disp}).pickle")
    y_va = load(f"{folder}/y_val({disp}).pickle")
    X_te = load(f"{folder}/X_{img}_test({disp}).pickle")
    y_te = load(f"{folder}/y_test({disp}).pickle")
    return X_tr, y_tr, X_va, y_va, X_te, y_te

## (Opcional) Inspecionar blocos DepthwiseConv2D da MobileNetV3Large

Lista todos os blocos depthwise — note que agora os **últimos** são os que serão substituídos. Eles operam em feature maps menores (7×7 ou 14×14) com muitos canais, o que torna a FFT barata e o filtro espectral expressivo.

In [ ]:
# Carrega so para inspecao - nao treina ainda
_img0 = IMAGENS[0]
_dev0 = DISPOSITIVOS[0]
_X_sample = pickle.load(open(f"{FOLDER_I}/X_{_img0}_train({_dev0}).pickle", 'rb'))
_input_shape = _X_sample.shape[1:]
del _X_sample

_base = MobileNetV3Large(input_shape=_input_shape, weights='imagenet', include_top=False)
dw_info = [
    (i, l.name, l.get_config()['strides'], l.output.shape)
    for i, l in enumerate(_base.layers)
    if isinstance(l, DepthwiseConv2D)
]
print(f"Input shape detectado: {_input_shape}")
print(f"Total de blocos DepthwiseConv2D: {len(dw_info)}")
print(f"{'idx':>5}  {'nome':<45}  {'strides':<10}  output_shape")
print('-' * 95)
for idx, name, strides, oshape in dw_info:
    print(f"{idx:>5}  {name:<45}  {str(strides):<10}  {oshape}")

max_n = max(VARIANTS.values())
if max_n > 0:
    print(f"\n>>> Variantes substituirao os ULTIMOS {max_n} blocos (em destaque acima).")
del _base

## Loop de experimentos

Para cada variante × dispositivo × tipo de imagem:
- Treina `N_RUNS` vezes (mesmo setup do notebook 1)
- Registra acurácia, F1-score, latência e tamanho do modelo

In [ ]:
os.makedirs('output', exist_ok=True)
results = []

for variant_name, n_spectral in VARIANTS.items():
    print(f"\n{'='*65}")
    print(f"VARIANTE: {variant_name}  ({n_spectral} blocos espectrais no final)")
    print(f"{'='*65}")

    for disp in DISPOSITIVOS:
        for img in IMAGENS:

            X_tr, y_tr, X_va, y_va, X_te, y_te = load_data(disp, img)
            input_shape = X_tr.shape[1:]

            run_accs, run_f1s, latencies = [], [], []
            size_mb, n_train, n_frozen = None, None, None

            for run in range(N_RUNS):
                ckpt = f'ckpt_dft_inv/{variant_name}/{disp}/{img}/run{run}/model.keras'
                os.makedirs(os.path.dirname(ckpt), exist_ok=True)

                model = build_full_model(input_shape, n_spectral)

                # Registra metadados uma vez por combinacao
                if run == 0:
                    size_mb  = model_size_mb(model)
                    n_train, n_frozen = count_params(model)

                model.fit(
                    X_tr, y_tr,
                    batch_size=BATCH,
                    epochs=100,
                    verbose=0,
                    validation_data=(X_va, y_va),
                    callbacks=[
                        EarlyStopping(
                            monitor='val_loss', patience=7,
                            restore_best_weights=False, verbose=0
                        ),
                        ModelCheckpoint(
                            ckpt, monitor='val_accuracy',
                            save_best_only=True, verbose=0
                        )
                    ]
                )

                best = tf.keras.models.load_model(
                    ckpt,
                    custom_objects={
                        'SpectralMultiplyDepthwise': SpectralMultiplyDepthwise
                    }
                )

                # Metricas
                preds = (best.predict(X_te, batch_size=BATCH, verbose=0) > 0.5).astype(int).flatten()
                run_accs.append(float(np.mean(preds == y_te)))
                run_f1s.append(float(sklearn_f1(y_te, preds, average='macro')))

                # Latencia num batch representativo
                x_sample = X_te[:BATCH]
                latencies.append(measure_latency_ms(best, x_sample))

                del model, best
                K.clear_session()

            row = {
                'variant':          variant_name,
                'n_spectral':       n_spectral,
                'device':           disp,
                'image':            img,
                'acc_mean':         np.mean(run_accs),
                'acc_std':          np.std(run_accs),
                'f1_mean':          np.mean(run_f1s),
                'f1_std':           np.std(run_f1s),
                'latency_ms_mean':  np.mean(latencies),
                'latency_ms_std':   np.std(latencies),
                'model_size_mb':    size_mb,
                'trainable_params': n_train,
                'frozen_params':    n_frozen,
            }
            results.append(row)

            print(
                f"  {disp:<16} | {img:<5} | "
                f"acc={np.mean(run_accs):.3f}±{np.std(run_accs):.3f} | "
                f"f1={np.mean(run_f1s):.3f} | "
                f"lat={np.mean(latencies):.1f}ms"
            )

df_results = pd.DataFrame(results)
df_results.to_csv('output/dft_inverse_experiment_results.csv', index=False)
print("\nResultados salvos -> output/dft_inverse_experiment_results.csv")
df_results.head()

## Resumo comparativo entre variantes

In [ ]:
summary = (
    df_results
    .groupby('variant')[['acc_mean', 'f1_mean', 'latency_ms_mean', 'model_size_mb', 'trainable_params']]
    .mean()
    .round(4)
)
print(summary.to_string())

## Visualizações

### Heatmaps de acurácia (dispositivo × tipo de imagem)

In [ ]:
import matplotlib.pyplot as plt

n_variants = len(VARIANTS)
fig, axes = plt.subplots(1, n_variants, figsize=(6 * n_variants, 5))
if n_variants == 1:
    axes = [axes]

for ax, variant_name in zip(axes, VARIANTS):
    sub = df_results[df_results['variant'] == variant_name]
    pivot = sub.pivot(index='device', columns='image', values='acc_mean')
    im = ax.imshow(pivot.values, vmin=0.5, vmax=1.0, cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(f'{variant_name}\n(acuracia media)')
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=8, color='black' if val < 0.8 else 'white')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('output/dft_inverse_heatmaps_accuracy.png', dpi=150)
plt.show()

### Comparação de latência, acurácia e F1

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

grouped = df_results.groupby('variant')

# Latencia
lat = grouped['latency_ms_mean'].mean()
axes[0].bar(lat.index, lat.values, color=['steelblue', 'darkorange', 'green'])
axes[0].set_title('Latencia media (ms)')
axes[0].set_ylabel('ms / batch')
axes[0].tick_params(axis='x', rotation=15)

# Acuracia media
acc = grouped['acc_mean'].mean()
axes[1].bar(acc.index, acc.values, color=['steelblue', 'darkorange', 'green'])
axes[1].set_title('Acuracia media')
axes[1].set_ylim(0.5, 1.0)
axes[1].tick_params(axis='x', rotation=15)

# F1 medio
f1 = grouped['f1_mean'].mean()
axes[2].bar(f1.index, f1.values, color=['steelblue', 'darkorange', 'green'])
axes[2].set_title('F1-score medio')
axes[2].set_ylim(0.5, 1.0)
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('output/dft_inverse_benchmark_summary.png', dpi=150)
plt.show()

print("\nLatencia por variante (ms):\n", lat.round(2).to_string())
print("\nAcuracia por variante:\n", acc.round(4).to_string())
print("\nF1-score por variante:\n", f1.round(4).to_string())

### Delta de acurácia: variantes espectrais vs. baseline

In [ ]:
baseline_acc = df_results[df_results['variant'] == 'baseline'].set_index(['device', 'image'])['acc_mean']

for v in [k for k in VARIANTS if k != 'baseline']:
    v_acc = df_results[df_results['variant'] == v].set_index(['device', 'image'])['acc_mean']
    delta = (v_acc - baseline_acc).unstack('image')
    print(f"\n--- Delta acc: {v} - baseline ---")
    print(delta.round(4).to_string())

### Comparação com o experimento 4 (DFT no início)

Se o CSV `output/dft_experiment_results.csv` do notebook 4 estiver disponível, podemos comparar lado a lado as duas estratégias.

In [ ]:
prev_csv = 'output/dft_experiment_results.csv'
if os.path.exists(prev_csv):
    df_prev = pd.read_csv(prev_csv)
    prev_summary = (
        df_prev
        .groupby('variant')[['acc_mean', 'f1_mean', 'latency_ms_mean']]
        .mean().round(4)
    )
    new_summary = (
        df_results
        .groupby('variant')[['acc_mean', 'f1_mean', 'latency_ms_mean']]
        .mean().round(4)
    )
    print('=== Experimento 4: DFT no INICIO (sem parametros aprendiveis) ===')
    print(prev_summary.to_string())
    print('\n=== Experimento 5: mult. espectral no FINAL (com filtro aprendivel) ===')
    print(new_summary.to_string())
else:
    print(f'CSV anterior nao encontrado em {prev_csv}; pulando comparacao.')